In [1]:
import ee

# Authenticate and initialize the Earth Engine API
ee.Authenticate()
ee.Initialize()


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [3]:
# !pip install geemap

import ee
import geemap
import pandas as pd

ee.Initialize()

# Load UC boundaries
uc_asset = "projects/ee-ahmedabclr35/assets/Lahore_union_Council_Boundries"
ucs = ee.FeatureCollection(uc_asset)

# Define time range
start = '2024-01-01'
end   = '2024-12-31'

# Sentinel-2 Surface Reflectance, filter + select only B4, B8
s2 = (ee.ImageCollection("COPERNICUS/S2_SR")
        .filterDate(start, end)
        .filterBounds(ucs)
        .select(['B4', 'B8'])  # ✅ keep only bands we need
        .median())


ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Reduce to UC level
uc_stats = ndvi.reduceRegions(
    collection=ucs,
    reducer=ee.Reducer.mean(),
    scale=10
)

# --- Option 1: Export locally as GeoJSON ---
geemap.ee_export_vector(uc_stats, filename="UC_NDVI.geojson")

# --- Option 2: Export locally as CSV ---
df = geemap.ee_to_pandas(uc_stats)
df.to_csv("UC_NDVI.csv", index=False)

print("Exported UC stats locally!")


Generating URL ...
Please wait ...
Data downloaded to d:\LUMS\Senior Fall 25\SPROJ - Dr Tahir\SPROJ\notebooks\UC_NDVI.geojson


AttributeError: module 'geemap' has no attribute 'ee_to_pandas'

In [5]:
import ee, geemap, pandas as pd
ee.Initialize()

# Get all features into Python dict
features = uc_stats.getInfo()['features']

# Flatten into list of dicts
rows = [f['properties'] for f in features]

# Convert to DataFrame
df = pd.DataFrame(rows)

# Save locally
df.to_csv("UC_NDVI.csv", index=False)
print("Saved UC NDVI locally!")


Saved UC NDVI locally!


In [9]:
import geopandas as gpd
import pandas as pd
import folium

# Load UC boundaries (GeoJSON)
uc_gdf = gpd.read_file("../data/Union_Councils.geojson")

# Load stats (CSV from GEE export)
stats_df = pd.read_csv("UC_NDVI.csv")

merged = uc_gdf.merge(stats_df, on="Name_UC_N")  

# Initialize folium map
m = folium.Map(location=[31.5204, 74.3587], zoom_start=11, tiles="cartodbpositron")

# Choropleth map (e.g. NDVI mean)
folium.Choropleth(
    geo_data=merged,
    data=merged,
    columns=["Name_UC_N", "mean"],   # UC name and the value to color
    key_on="feature.properties.Name_UC_N",
    fill_color="YlGn",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="NDVI Mean"
).add_to(m)

# Add tooltips (hover labels)
folium.GeoJson(
    merged,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    tooltip=folium.GeoJsonTooltip(
        fields=["Name_UC_N", "mean", "P_D_16_x"],  # what to show on hover
        aliases=["UC Name:", "NDVI Mean:", "Pop Density 2016:"],
        localize=True
    )
).add_to(m)

# Save to HTML
m.save("UC_NDVI_heatmap.html")